# QueryKFold example

`QueryKFold` is a K-fold splitter for Learning-to-Rank data represented as `(X, y, q)`, where `q` contains the query id for each document.

Unlike a standard row-based KFold, the split is built query by query: for each query, one chunk of its documents goes to the test split and the remaining chunks go to the train split. Therefore, the same query can appear in both train and test in the same fold, but the same document never appears in both splits.

In [1]:
import numpy as np
import pandas as pd

from ltr_utility.model_selection.query_kfold import QueryKFold


## Toy dataset

Each row is one document. Documents with the same `query_id` belong to the same ranking list.

In [2]:
q = np.array([100, 100, 100, 100, 100, 200, 200, 200, 300, 300, 300, 300, 300])
doc_id = np.arange(len(q))

# Two simple features: the document id and the query id.
# The first column makes it easy to check that no document leaks across splits.
X = np.column_stack([doc_id, q])
y = np.array([4, 3, 2, 1, 0, 2, 4, 1, 0, 1, 3, 4, 2])

dataset = pd.DataFrame({
    "doc_id": doc_id,
    "query_id": q,
    "target": y,
    "feature_0": X[:, 0],
    "feature_1": X[:, 1],
})

dataset


,doc_id,query_id,target,feature_0,feature_1
0,0,100,4,0,100
1,1,100,3,1,100
2,2,100,2,2,100
3,3,100,1,3,100
4,4,100,0,4,100
5,5,200,2,5,200
6,6,200,4,6,200
7,7,200,1,7,200
8,8,300,0,8,300
9,9,300,1,9,300


In [3]:
query_sizes = (
    dataset
    .groupby("query_id")
    .size()
    .rename("documents")
    .to_frame()
)

query_sizes


,documents
query_id,
100,5
200,3
300,5


## Split by query

The output of `split` is a pair of tuples: `(X_train, y_train, q_train)` and `(X_test, y_test, q_test)`.

In [4]:
qkf = QueryKFold(n_splits=3)

fold_tables = []
fold_checks = []

for fold_idx, (train_data, test_data) in enumerate(qkf.split(X, y, q), start=1):
    X_train, y_train, q_train = train_data
    X_test, y_test, q_test = test_data

    train_doc_ids = X_train[:, 0].astype(int)
    test_doc_ids = X_test[:, 0].astype(int)
    shared_doc_ids = set(train_doc_ids).intersection(test_doc_ids)

    fold_table = pd.concat([
        pd.DataFrame({
            "fold": fold_idx,
            "split": "train",
            "doc_id": train_doc_ids,
            "query_id": q_train,
            "target": y_train,
        }),
        pd.DataFrame({
            "fold": fold_idx,
            "split": "test",
            "doc_id": test_doc_ids,
            "query_id": q_test,
            "target": y_test,
        }),
    ], ignore_index=True).sort_values(["query_id", "split", "doc_id"])

    fold_tables.append(fold_table)
    fold_checks.append({
        "fold": fold_idx,
        "train documents": len(train_doc_ids),
        "test documents": len(test_doc_ids),
        "shared documents": len(shared_doc_ids),
        "train queries": sorted(np.unique(q_train).tolist()),
        "test queries": sorted(np.unique(q_test).tolist()),
    })

fold_summary = pd.DataFrame(fold_checks)
fold_summary


,fold,train documents,test documents,shared documents,train queries,test queries
0,1,8,5,0,"[100, 200, 300]","[100, 200, 300]"
1,2,8,5,0,"[100, 200, 300]","[100, 200, 300]"
2,3,10,3,0,"[100, 200, 300]","[100, 200, 300]"


In [5]:
query_distribution = pd.concat([
    (
        fold_table
        .groupby(["fold", "query_id", "split"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["train", "test"], fill_value=0)
        .reset_index()
    )
    for fold_table in fold_tables
], ignore_index=True)

query_distribution


split,fold,query_id,train,test
0,1,100,3,2
1,1,200,2,1
2,1,300,3,2
3,2,100,3,2
4,2,200,2,1
5,2,300,3,2
6,3,100,4,1
7,3,200,2,1
8,3,300,4,1


## Fold contents

The table below shows the actual documents assigned to train and test for each fold.

In [6]:
fold_contents = (
    pd.concat(fold_tables, ignore_index=True)
    .sort_values(["fold", "query_id", "split", "doc_id"])
    .reset_index(drop=True)
)

fold_contents


,fold,split,doc_id,query_id,target
0,1,test,0,100,4
1,1,test,1,100,3
2,1,train,2,100,2
3,1,train,3,100,1
4,1,train,4,100,0
5,1,test,5,200,2
6,1,train,6,200,4
7,1,train,7,200,1
8,1,test,8,300,0
9,1,test,9,300,1


## Sanity checks

These checks verify the two key properties of the splitter: every fold covers the full dataset through `train + test`, and no document id is shared between train and test in the same fold.

In [7]:
assert fold_summary["shared documents"].eq(0).all()
assert all(len(fold_table) == len(dataset) for fold_table in fold_tables)

print("All checks passed.")


All checks passed.
